# MCP Chatbot

A LangGraph chatbot that dynamically loads tools from multiple MCP servers (a local stdio math server and a remote HTTP expense tracker).

## 1. Install required packages

In [8]:
pip install -qU langgraph langchain-openai langchain-mcp-adapters python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Setup & imports

In [9]:
import os
from typing import Annotated

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY not found. Add it to a .env file in this notebook's directory."
    )

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini")

## 3. Configure MCP servers

- `math_server` — local stdio server in `mcp-math-server/main.py` (next to this notebook).
- `expense_tracker` — a remote demo server. Replace the URL with your own if you have one; the graph-building step below skips any server that's unreachable instead of failing entirely.

In [10]:
import sys

THIS_DIR = os.getcwd()

client = MultiServerMCPClient(
    {
        # 1. Local MCP server (math calculator via stdio)
        "math_server": {
            "command": sys.executable,
            "args": [os.path.join(THIS_DIR, "mcp-math-server", "main.py")],
            "transport": "stdio",
        },
        # 2. Remote MCP server (expense tracker via streamable HTTP)
        # Replace with your own server if this demo URL is unreachable.
        "expense_tracker": {
            "url": "https://expense-tracker-mcp.onrender.com/sse",
            "transport": "streamable_http",
        },
    }
)

## 4. Define graph state

In [11]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

## 5. Build the graph (dynamic MCP tool loading)

In [12]:
async def load_mcp_tools():
    """Fetch tools from each MCP server independently, so one unreachable
    server (e.g. a dead demo endpoint) doesn't wipe out the others."""
    tools = []
    for server_name in client.connections:
        try:
            server_tools = await client.get_tools(server_name=server_name)
            tools.extend(server_tools)
            print(f"Loaded {len(server_tools)} tool(s) from '{server_name}': {[t.name for t in server_tools]}")
        except Exception as exc:
            print(f"Skipping '{server_name}': {exc}")
    return tools


async def build_graph():
    mcp_tools = await load_mcp_tools()

    llm_with_tools = llm.bind_tools(mcp_tools) if mcp_tools else llm

    async def chat_node(state: State):
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    builder = StateGraph(State)
    builder.add_node("chat_node", chat_node)

    if mcp_tools:
        tool_node = ToolNode(mcp_tools)
        builder.add_node("tools", tool_node)
        builder.add_edge(START, "chat_node")
        builder.add_conditional_edges("chat_node", tools_condition)
        builder.add_edge("tools", "chat_node")
    else:
        builder.add_edge(START, "chat_node")
        builder.add_edge("chat_node", END)

    return builder.compile()


chatbot = await build_graph()

Loaded 4 tool(s) from 'math_server': ['add', 'subtract', 'multiply', 'divide']
Skipping 'expense_tracker': unhandled errors in a TaskGroup (1 sub-exception)


## 6. Run example queries

In [13]:
# Example 1: add an expense using the expense-tracker MCP server
prompt = "Add an expense of 500 for a Udemy course on 10th November."
response = await chatbot.ainvoke({"messages": [("user", prompt)]})
print("\n--- Response 1 ---")
print(response["messages"][-1].content)


--- Response 1 ---
It looks like you're trying to record an expense. Typically, tracking expenses involves maintaining a ledger or a personal finance application. 

To summarize your request:
- **Expense Amount**: 500
- **Description**: Udemy course
- **Date**: 10th November

If you need assistance with financial calculations or a summary of your expenses, please let me know! Otherwise, I would recommend entering this information in your preferred tracking tool.


In [14]:
# Example 2: list expenses using the expense-tracker MCP server
prompt2 = "Give me all my expenses for the month of November from 1st November to 30th November."
response2 = await chatbot.ainvoke({"messages": [("user", prompt2)]})
print("\n--- Response 2 ---")
print(response2["messages"][-1].content)


--- Response 2 ---
I currently don't have access to your personal expenses or financial records. You can check your expenses for November through your bank statement, budgeting app, or financial software. If you need help organizing or calculating expenses, please provide the necessary details, and I can assist with that!
